# Network size and seed on the frozen magnet crossing

This notebook **loads** the tables written by `aggregate.py` and the figures written by
`plot.py`. It recomputes nothing: every number below comes from `results/summary.csv`
and `results/by_architecture.csv`, which in turn come from the per-run json records
that `_shared/train.py` wrote on the farm.

The experiment is the verified baseline of `../One_step_network_v2` — the paper's
one-step discrete-time network on the frozen magnet crossing, q = 8 Gauss-Legendre
stages, MagDown, physics loss, trained to genuine stall and then confirmed — with two
things varied and nothing else: the **network size** (widths 32, 50, 100, 200 x depths
2, 4, 6) and the **seed** (0-9). 120 runs.

Three questions:

1. **Capacity or optimisation?** The exact scheme itself lands 22.5 um from the fp64 RK4
   reference on these same 2018 test states, so ~23 um is a hard floor for any network
   trained on it. The baseline sits at ~180 um, eight times above it. If the floor is a
   *capacity* limit, bigger networks should walk it down; if it is an *optimisation*
   limit, the final training loss should predict the endpoint error and size buys little.
2. **How much does the seed matter?** Ten seeds per architecture measure the spread that
   the baseline's three seeds could only hint at.
3. **How do we pick a run?** If validation endpoint error ranks the seeds the same way
   test endpoint error does, we can select on validation honestly.

In [ ]:
import json
import os
import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from IPython.display import Image, display

RESULTS, FIGURES = "results", "figures"
CEILING_UM = json.load(open('../Stage_count_sweep/results/'
                            'scheme_ceiling_same_population_q08.json'))['endpoint_med_um']
print('exact-scheme ceiling on these 2018 test states: %.1f um' % CEILING_UM)

runs = pd.read_csv(os.path.join(RESULTS, "summary.csv"))
arch = pd.read_csv(os.path.join(RESULTS, "by_architecture.csv"))
print("%d run records: %d converged, %d not"
      % (len(runs), runs.converged.sum(), (~runs.converged).sum()))
runs.head()

## 1. Completion: which runs converged

A run counts as converged only if it stalled (two consecutive restarts each improving the loss by less than 1%) *and* a fresh optimiser re-stalled within two restarts without moving the endpoint medians. Anything else is reported as unconverged and never pooled with the converged runs.

In [ ]:
grid = (runs[runs['mode'] == 'physics']
        .pivot_table(index='depth', columns='width', values='converged',
                     aggfunc='sum'))
counts = (runs[runs['mode'] == 'physics']
          .pivot_table(index='depth', columns='width', values='seed', aggfunc='count'))
print('converged runs per architecture (of the runs that have finished):')
display(grid.astype('Int64'))
print('runs finished per architecture (of 10):')
display(counts.astype('Int64'))

## 2. The floor against network size

One row per architecture: the median over the converged seeds of the test endpoint error, its min and max over those seeds, and the cost of getting there.

In [ ]:
cols = ['mode', 'depth', 'width', 'n_params', 'n_runs', 'n_converged',
        'test_endpoint_med_um_median', 'test_endpoint_med_um_min',
        'test_endpoint_med_um_max', 'test_endpoint_spread_ratio',
        'val_endpoint_med_um_median', 'restarts_median', 'wall_s_median',
        'final_loss_median']
display(arch[cols].style.format({c: '{:.4g}' for c in cols
                                 if arch[c].dtype.kind == 'f'}))

In [ ]:
display(Image(os.path.join(FIGURES, 'floor_vs_architecture.png')))

### How far did the floor move?

Compared against the exact-scheme ceiling (22.5 um on these very test states, measured by
`../Stage_count_sweep/measure_scheme_ceiling.py`) and against the baseline's 4x50.

In [ ]:
phys = arch[(arch['mode'] == 'physics') & (arch['n_converged'] > 0)]
best = phys.loc[phys['test_endpoint_med_um_median'].idxmin()]
worst = phys.loc[phys['test_endpoint_med_um_median'].idxmax()]
base = phys[(phys.width == 50) & (phys.depth == 4)]
print('best  architecture : %dx%d (%d params), median test %.1f um'
      % (best.depth, best.width, best.n_params, best.test_endpoint_med_um_median))
print('worst architecture : %dx%d (%d params), median test %.1f um'
      % (worst.depth, worst.width, worst.n_params, worst.test_endpoint_med_um_median))
if len(base):
    b = base.iloc[0]
    print('baseline 4x50      : median test %.1f um' % b.test_endpoint_med_um_median)
    print('best / baseline    : %.2f' % (best.test_endpoint_med_um_median
                                         / b.test_endpoint_med_um_median))
print('best / ceiling     : %.1f x the %g um exact-scheme ceiling'
      % (best.test_endpoint_med_um_median / CEILING_UM, CEILING_UM))

### Does depth hurt?

On van der Pol, depth 6 was worse than depth 4 at the same width — the extra layers made
the L-BFGS problem harder without adding anything the task needed. The same comparison here,
at fixed width.

In [ ]:
piv = (arch[arch['mode'] == 'physics']
       .pivot_table(index='width', columns='depth',
                    values='test_endpoint_med_um_median'))
display(piv.style.format('{:.1f}'))
print('depth 6 / depth 4 at each width:')
if 6 in piv.columns and 4 in piv.columns:
    print((piv[6] / piv[4]).round(2).to_string())

## 3. Optimisation or capacity: the loss-error correlation

If the runs sit on a single loss-error curve, the endpoint error is whatever the optimiser managed to leave on the table, and lowering the loss further is the only thing that helps. On van der Pol this correlation was +0.97.

In [ ]:
display(Image(os.path.join(FIGURES, 'loss_vs_error.png')))
c = runs[runs.converged & (runs['mode'] == 'physics')]
r = spearmanr(c.final_loss, c.test_endpoint_med_um)
print('Spearman rho(final loss, test endpoint error) = %.3f  (p = %.2g, n = %d)'
      % (r.statistic, r.pvalue, len(c)))

## 4. The seed spread

Ten seeds per architecture: the same code, the same data, the same protocol, only a different initialisation.

In [ ]:
display(Image(os.path.join(FIGURES, 'seed_spread.png')))
sp = (arch[(arch['mode'] == 'physics') & (arch.n_converged > 1)]
      [['depth', 'width', 'n_converged', 'test_endpoint_med_um_min',
        'test_endpoint_med_um_max', 'test_endpoint_spread_ratio']]
      .sort_values('test_endpoint_spread_ratio'))
display(sp.style.format({'test_endpoint_med_um_min': '{:.1f}',
                         'test_endpoint_med_um_max': '{:.1f}',
                         'test_endpoint_spread_ratio': '{:.2f}'}))
print('median max/min over architectures: %.2f'
      % sp.test_endpoint_spread_ratio.median())

## 5. The selection rule: does validation pick the seed?

We are allowed to look at validation when choosing a run; test is the number we then report. The question is whether validation ranks the seeds the way test does.

In [ ]:
display(Image(os.path.join(FIGURES, 'val_vs_test_selection.png')))
c = runs[runs.converged & (runs['mode'] == 'physics')]
print('overall Spearman rho(val, test) = %.3f over %d runs'
      % (spearmanr(c.val_endpoint_med_um, c.test_endpoint_med_um).statistic, len(c)))

rows = []
for (w, d), g in c.groupby(['width', 'depth']):
    if len(g) < 3:
        continue
    pick = g.loc[g.val_endpoint_med_um.idxmin()]
    bestr = g.loc[g.test_endpoint_med_um.idxmin()]
    rows.append({'depth': d, 'width': w, 'n': len(g),
                 'rho_val_test': spearmanr(g.val_endpoint_med_um,
                                           g.test_endpoint_med_um).statistic,
                 'val_pick_seed': int(pick.seed), 'val_pick_test_um': pick.test_endpoint_med_um,
                 'best_seed': int(bestr.seed), 'best_test_um': bestr.test_endpoint_med_um,
                 'penalty': pick.test_endpoint_med_um / bestr.test_endpoint_med_um})
sel = pd.DataFrame(rows).sort_values(['width', 'depth'])
display(sel.style.format({'rho_val_test': '{:.2f}', 'val_pick_test_um': '{:.1f}',
                          'best_test_um': '{:.1f}', 'penalty': '{:.2f}'}))
print('validation-best seed is also test-best in %d of %d architectures'
      % ((sel.val_pick_seed == sel.best_seed).sum(), len(sel)))
print('median penalty for selecting on validation: %.2f x' % sel.penalty.median())

## 6. The data twin at the winning architecture

Ten fresh supervised runs (MSE against the fp64 RK4 stage and endpoint labels) at the architecture the physics grid chose. The v2 data-twin runs were four-thread and are not bit-comparable, so all ten were rerun here on one thread.

In [ ]:
twin = runs[runs['mode'] == 'data']
if len(twin):
    display(twin[['tag', 'seed', 'width', 'depth', 'converged', 'restarts',
                  'final_loss', 'val_endpoint_med_um', 'test_endpoint_med_um',
                  'test_endpoint_p95_um']]
            .style.format({'final_loss': '{:.3g}', 'val_endpoint_med_um': '{:.1f}',
                           'test_endpoint_med_um': '{:.1f}',
                           'test_endpoint_p95_um': '{:.0f}'}))
    tc = twin[twin.converged]
    pc = runs[(runs['mode'] == 'physics') & runs.converged
              & (runs.width == twin.width.iloc[0]) & (runs.depth == twin.depth.iloc[0])]
    print('data twin  : median test %.1f um over %d converged seeds'
          % (tc.test_endpoint_med_um.median(), len(tc)))
    print('physics     : median test %.1f um over %d converged seeds'
          % (pc.test_endpoint_med_um.median(), len(pc)))
    print('physics / data = %.2f' % (pc.test_endpoint_med_um.median()
                                     / tc.test_endpoint_med_um.median()))
else:
    print('no data-twin runs recorded yet')

## 7. Verdict

See `README.md` for the four statements this study is asked to settle: capacity vs optimisation, whether depth 6 hurts, the architecture recommended for the general-leg experiment, and the selection rule.